## Ingestion of regression data from public sources
____
This is where we are going to collect all the features and targets we are going to use for the multimodal model. Our features include on the ground atmospheric data, satellite imagery for days with smoke from either industry or wildfires. The target will be on the ground readings of PM2.5 measures.

### 1. Gather and put all on-the-ground readings into a single table seperated by site

- CSV files downloaded from EPA AirData portal located in src/ingestion/raw_data
- Study Area: California
- Study period: 2018-01-01 - 2025-12-31
- _Source_: https://rconnect-public.epa.gov/AirData/#shiny-tab-DailySummary

In [2]:
import pandas as pd
from pathlib import Path
import requests
from config import settings

SOURCE_DIR = settings.raw_data

directory = Path(SOURCE_DIR)
df = []

for item in directory.glob("*.csv"):
    data = pd.read_csv(item)
    df.append(data)
df = pd.concat(df, ignore_index=True)
df.to_csv("combined_pm25_readings.csv") # GATE 1


### 2. Clean the data so there are no duplicates, no null values, and all the data types are the same throughout

- Some duplicates are from the same site on the same day but with different sensors. Will make the assumption that the most reliable one is the lowest.
- Others have unexplained duplicates that will be diagnosed

In [ ]:
EPA_KEY = settings.epa_key
BASE_URL = settings.base_url
def test_suspicious_dates(date: str):    
    PARAMS = {
        "email": settings.epa_user,
        "key": settings.epa_key,
        "param": 88101,
        "state": "06", "county": "001", "site": "0007",
        "bdate": date.replace('-', ""), "edate": date.replace('-', ""),
    }
    mask = (df["Site ID"] == 60010007) & (df["Date"] == date)
    pm25_val = df.loc[mask, "Daily Mean PM2.5 Concentration"]
    livermore_api_data = requests.get(f"{BASE_URL}dailyData/bySite", params=PARAMS)
    print(f"--------------{date}--------------------")
    print("--------------API SCORES-----------------", end='\n')
    print([d["arithmetic_mean"] for d in livermore_api_data.json().get("Data", [])], end='\n')
    print("--------------DOWNLOAD DATA--------------", end='\n')
    print(f"Download Data PM2.5: {pm25_val.item()}", end='\n\n')

test_suspicious_dates("2018-01-01")
test_suspicious_dates("2018-01-02")
test_suspicious_dates("2018-01-03")
test_suspicious_dates("2018-01-04")
test_suspicious_dates("2018-01-05")

--------------API SCORES-----------------
[28.166667, 28.1, 28.1, 28.1, 28.1, 28.1, 28.1, 28.1, 28.1]
--------------DOWNLOAD DATA--------------
Download Data PM2.5: 24.5

--------------API SCORES-----------------
[24.5, 24.5, 24.5, 24.5, 24.5, 24.5, 24.5, 24.5, 24.5]
--------------DOWNLOAD DATA--------------
Download Data PM2.5: 39.0

--------------API SCORES-----------------
[39.086957, 39.0, 39.0, 39.0, 39.0, 39.0, 39.0, 39.0, 39.0]
--------------DOWNLOAD DATA--------------
Download Data PM2.5: 20.9

--------------API SCORES-----------------
[20.916667, 20.9, 20.9, 20.9, 20.9, 20.9, 20.9, 20.9, 20.9]
--------------DOWNLOAD DATA--------------
Download Data PM2.5: 5.5

--------------API SCORES-----------------
[5.583333, 5.5, 5.5, 5.5, 5.5, 5.5, 5.5, 5.5, 5.5]
--------------DOWNLOAD DATA--------------
Download Data PM2.5: 6.6

